# 1. Preprocessing Overview

**Project:** Predicting Rural Water Point Functionality Using Data Mining. **Target:** `functional3`, a three-class nominal classification outcome.

This notebook is **Step 5: Data Preprocessing, Feature Engineering and Feature Selection**. The raw Excel dataset is read-only. It preserves fixed data-quality and leakage rules from the EDA and decision plan, while defining reusable candidate preprocessing strategies. No strategy is declared optimal here.

```text
Raw data → fixed predictor exclusions → locked stratified train/holdout split
         → fixed cleaning → candidate preprocessing
                            ├─ scale-sensitive
                            ├─ tree/boosting (one-hot categories)
                            └─ CatBoost native (categorical strings)
         → structural validation only
Notebook 04: training-only CV → compare preprocessing + model combinations
Notebook 05: optimize shortlisted combinations → one-time holdout evaluation
```

Notebook 03 does **not** train or tune classifiers, compare predictive scores, resample data, select a final model, or use the holdout to choose preprocessing. Every learned preprocessing statistic is fitted on development training rows only during this structural demonstration. In later cross-validation, the *complete* cleaner + feature engineer + preprocessor + classifier pipeline must be refitted inside each training fold. The holdout is reserved for final evaluation.
CatBoost-native candidates retain nominal categories as strings for CatBoost to process internally. CatBoost itself is neither imported nor trained here.


# 2. Load Raw Dataset

Load the raw Excel file exactly as in the previous two notebooks. The file is treated as **read-only**: every subsequent step operates on in-memory copies (`df.copy()`, or new `DataFrame`s produced by `.drop()` / transformers), and the original file on disk is never written to.

In [37]:
import hashlib
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

DATA_PATH = "../data/raw/water_pump_functionality.xlsx"
TARGET = "functional3"
CLASS_ORDER = ["Functional", "Partially functional", "Abandoned or not functional"]
RANDOM_STATE = 42
TEST_SIZE = 0.20

def file_digest(path):
    with open(path, "rb") as source:
        return hashlib.sha256(source.read()).hexdigest()

_raw_digest_before = file_digest(DATA_PATH)
LEGACY_PATHS = [
    "../data/processed/X_train_processed.csv", "../data/processed/X_test_processed.csv",
    "../data/processed/y_train.csv", "../data/processed/y_test.csv",
    "../data/processed/feature_names.csv", "../models/preprocessing_pipeline.joblib",
]
_legacy_digests_before = {path: file_digest(path) if os.path.exists(path) else None for path in LEGACY_PATHS}
notebook_cells_for_validation = json.loads(Path("03_preprocessing.ipynb").read_text(encoding="utf-8"))["cells"]
df_raw = pd.read_excel(DATA_PATH)
print(f"Raw dataset loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head()

Raw dataset loaded: 1793 rows, 52 columns


,id,country,admin1,admin2,admin3,instance_wp,subdate_wp,latitude_wp,longitude_wp,elevation_wp,...,qtyhh_c_wp,wc_admin_index_wp,wc_finance_index_wp,wc_mgmt_index_wp,wc_maint_index_wp,wc_savings_wp,wpqty_wp,pop_1000,annual_rain,season
0,1,Ethiopia,Tigray,North Western,Medebay Zana,762331022,2020-01-02,14.116315,38.493830,2033.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Ethiopia,Tigray,North Western,Tselemti,751461172,2019-12-21,13.581540,38.467207,1428.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Ethiopia,Tigray,North Western,Medebay Zana,758401118,2019-12-29,14.097803,38.452013,1997.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Ethiopia,Tigray,North Western,Medebay Zana,769951024,2019-12-29,13.912499,38.437931,1877.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Ethiopia,Tigray,North Western,Tahtay Koraro,765940926,2019-12-18,14.041615,38.437492,1966.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
# Work on a copy from this point on -- df_raw itself is never modified or re-saved.
df = df_raw.copy()
assert df_raw.shape == (1793, 52), "Unexpected raw shape -- verify the source file"
print("Copy created for preprocessing. df_raw remains untouched (read-only reference).")

Copy created for preprocessing. df_raw remains untouched (read-only reference).


# 3. Define Target and Features

`functional3` is the target (`y`). Every other raw column is a *candidate* predictor at this stage; leakage, identifier and administrative columns are removed explicitly in Section 4 before `X` is finalized.

In [39]:
assert TARGET in df.columns, f"Target column '{TARGET}' not found"

y_full = df[TARGET].copy()
X_full_candidates = df.drop(columns=[TARGET]).copy()

print(f"Target: '{TARGET}'")
print(f"Candidate predictor columns (before leakage/identifier removal): {X_full_candidates.shape[1]}")
print()
print("Target class counts (full raw dataset):")
print(y_full.value_counts().reindex(CLASS_ORDER))

Target: 'functional3'
Candidate predictor columns (before leakage/identifier removal): 51

Target class counts (full raw dataset):
functional3
Functional                     1333
Partially functional            157
Abandoned or not functional     303
Name: count, dtype: int64


# 4. Remove Leakage and Identifier Variables

This is a **fixed, rule-based** removal -- it uses no statistic computed from the data, so it is safe to apply before the train/locked holdout split (it cannot leak information between splits; it only removes columns that should never be model inputs at all).

### A. Target leakage variables (decision-plan Section 2)

These describe **current/recent water availability** -- i.e. they are symptoms or restatements of `functional3` itself, not independent predictors:

| Variable | Reason for exclusion |
|---|---|
| `wateravailable` | Near-perfect split of the target (EDA Sec. 12): "No" -> 300 Abandoned + 2 Partially + 0 Functional; "Yes" -> 1,333 Functional + 155 Partially + 3 Abandoned. |
| `whynowatertoday_wp` | Only ever filled in when there is currently no water -- its *missingness itself* almost perfectly reveals the target class (non-missing for 300/303 Abandoned, 2/157 Partially, 0/1,333 Functional). |
| `whynowatertoday_wp_other` | Free-text follow-up to the field above; same leakage mechanism, and 99.0% missing besides. |
| `downtime2weeks` | "Yes" -> 272 Abandoned + 81 Partially + 1 Functional; "No" -> 30 Abandoned + 76 Partially + 1,332 Functional -- a direct downtime symptom of the target. |
| `minutesfill20l` | Mean 1.45 min for Functional vs. 98.4 min for Partially functional; almost no data at all for Abandoned. A direct symptom of a struggling/non-functional source. |
| `brokendays_wp` | Mean 429.8 days (Abandoned) vs. 27.2 (Functional) vs. 50.0 (Partially functional) -- a quantified restatement of non-functionality. |
| `qtymonthsnowater_wp` | Mean 2.51 months (Abandoned) vs. 0.13 (Functional) vs. 0.53 (Partially functional) -- same reasoning. |

`lockedfullday_wp` was investigated in the EDA for the same reason and was **not** found to behave like a leakage variable (18.4% vs. 11.7% abandoned across its two levels -- no sharp separation), so it is kept as a candidate predictor.

### B. Identifier / survey-design / high-cardinality administrative variables (decision-plan Sections 2, 7-D)

| Variable | Reason for exclusion |
|---|---|
| `id` | Row-level identifier, 1,793/1,793 unique. Carries no generalizable signal. |
| `instance_wp` | Survey-instance identifier, 1,793/1,793 unique. |
| `photo_wp` | Photo URL/identifier, 1,792/1,793 unique. |
| `commid_wp` | Community identifier, 1,395/1,793 unique -- too fine-grained to generalize. |
| `cwid` | Committee identifier, 948 unique, 46.5% missing. |
| `timepoint_wp` | Encodes which *survey wave/sample* a record belongs to (a study-design artifact -- mean `wp_age` is 1.77 yrs for "new construction" vs. 5.87 yrs for "older construction" records), not a property of the water point itself. |
| `dataorg_wp` | Maps 1-to-1 onto `country` (every `ET.RST` record is Ethiopia, etc.) -- functionally redundant; `country` is kept instead. |
| `subdate_wp` | Survey submission date (338 unique date strings). Categorized under decision-plan Section 8, Category D (identifier / survey-design columns to exclude). It is not itself a property of the water point and no feature-engineering use for it was approved in the decision plan, so it is excluded here even though it is not individually re-listed in the Step 5 task brief -- flagged again in Section 17 (Unresolved Issues) for visibility. |
| `admin2`, `admin3` | High-cardinality administrative subdivisions (30 and 108 categories respectively, the latter 11.2% missing) relative to a 1,793-row dataset -- one-hot encoding `admin3` alone would add >100 columns. `country` and `admin1` (15 categories, 0% missing) are kept as the geographic predictors, per the task brief. |

### C. Columns flagged for further investigation, not yet cleared for inclusion (decision-plan Section 8, Category E)

`grant_number_wp` (69.5% missing; only 10 distinct values -- looks like a program code, not a continuous variable) and `balance_any_dollars_wp` (47.6% missing) are listed in the decision plan as **"requires further investigation before inclusion"**, separate from the approved predictor candidates in Category B. The Step 5 task brief does not instruct cleaning, engineering, or including either column. Per "do not silently introduce unrelated preprocessing decisions," they are **excluded here pending that investigation**, rather than silently included or silently dropped without a paper trail -- also flagged in Section 17.

All other candidate predictors are carried forward into the train/locked holdout split.

In [40]:
LEAKAGE_COLS = [
    "wateravailable",
    "whynowatertoday_wp",
    "whynowatertoday_wp_other",
    "downtime2weeks",
    "minutesfill20l",
    "brokendays_wp",
    "qtymonthsnowater_wp",
]

IDENTIFIER_ADMIN_COLS = [
    "id",
    "instance_wp",
    "photo_wp",
    "commid_wp",
    "cwid",
    "timepoint_wp",
    "dataorg_wp",
    "subdate_wp",
    "admin2",
    "admin3",
]

PENDING_INVESTIGATION_COLS = [
    "grant_number_wp",
    "balance_any_dollars_wp",
]

DROP_COLS = LEAKAGE_COLS + IDENTIFIER_ADMIN_COLS + PENDING_INVESTIGATION_COLS
assert len(DROP_COLS) == len(set(DROP_COLS)), "Duplicate column in drop list"

missing_from_df = [c for c in DROP_COLS if c not in df.columns]
assert not missing_from_df, f"Expected columns not found in raw data: {missing_from_df}"

y = df[TARGET].copy()
X = df.drop(columns=[TARGET] + DROP_COLS).copy()

print(f"Leakage columns removed: {len(LEAKAGE_COLS)}")
print(f"Identifier/administrative columns removed: {len(IDENTIFIER_ADMIN_COLS)}")
print(f"Pending-investigation columns removed: {len(PENDING_INVESTIGATION_COLS)}")
print(f"Total columns removed (excluding target): {len(DROP_COLS)}")
print(f"X shape after removal: {X.shape}")

assert TARGET not in X.columns
assert all(c not in X.columns for c in DROP_COLS)
print("\nVerified: target and all excluded columns are absent from X.")

Leakage columns removed: 7
Identifier/administrative columns removed: 10
Pending-investigation columns removed: 2
Total columns removed (excluding target): 19
X shape after removal: (1793, 32)

Verified: target and all excluded columns are absent from X.


# 5. Locked Train / Holdout Split

The historical `X_test`/`y_test` names are retained for compatibility; this is a **locked final holdout**, not a dataset for preprocessing selection. The split remains `test_size=0.20`, `random_state=42`, stratified on `functional3`. No learned preprocessing step is fitted before this split. `y_test` is used here only to verify stratification.

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")
assert X_train.shape[0] + X_test.shape[0] == X.shape[0]
assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0] == y_test.shape[0]
X_train_original = X_train.copy(deep=True)


X_train: (1434, 32), X_test: (359, 32)
y_train: (1434,), y_test: (359,)


In [42]:
# Class distribution: full dataset vs. train vs. test
dist_full = (y.value_counts(normalize=True) * 100).reindex(CLASS_ORDER).round(2)
dist_train = (y_train.value_counts(normalize=True) * 100).reindex(CLASS_ORDER).round(2)
dist_test = (y_test.value_counts(normalize=True) * 100).reindex(CLASS_ORDER).round(2)

count_full = y.value_counts().reindex(CLASS_ORDER)
count_train = y_train.value_counts().reindex(CLASS_ORDER)
count_test = y_test.value_counts().reindex(CLASS_ORDER)

class_dist_report = pd.DataFrame({
    "full_count": count_full, "full_pct": dist_full,
    "train_count": count_train, "train_pct": dist_train,
    "test_count": count_test, "test_pct": dist_test,
})
class_dist_report

,full_count,full_pct,train_count,train_pct,test_count,test_pct
functional3,,,,,,
Functional,1333,74.34,1066,74.34,267,74.37
Partially functional,157,8.76,126,8.79,31,8.64
Abandoned or not functional,303,16.90,242,16.88,61,16.99


**Observation:** stratification preserves similar class proportions in the development and locked holdout partitions. These counts verify the split; they are not predictive evaluation.

# 6. Fixed Deterministic Data Cleaning

`wc_savings_wp` survey codes `999` and `888` mean missing, not savings magnitudes. `FixedDataCleaner` makes this fixed rule the first step of every candidate pipeline. It uses no target or fitted statistic and returns a copy. Thus cross-validation folds, full training, holdout transformation, and eventual inference all apply the same rule. The preview below does not change `X_train` or `X_test`.

In [43]:
class FixedDataCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        cleaned = X.copy()
        cleaned["wc_savings_wp"] = cleaned["wc_savings_wp"].replace({999.0: np.nan, 888.0: np.nan})
        return cleaned

_clean_preview = FixedDataCleaner().fit_transform(X_train)
print("Training sentinel counts before:", X_train["wc_savings_wp"].isin([999, 888]).sum())
print("Training sentinel counts after:", _clean_preview["wc_savings_wp"].isin([999, 888]).sum())
assert X_train["wc_savings_wp"].equals(X_train.copy()["wc_savings_wp"])

Training sentinel counts before: 66
Training sentinel counts after: 0


## Duplicates and outliers

EDA found no fully duplicated rows. No observations are deleted or capped. Plausible extremes remain, and suspicious negative `elevation_wp` values remain unresolved pending a checked reference source. The scale-sensitive family's optional `log1p` transform handles selected non-negative skewed variables without deleting observations.

# 7. Missing-Value Strategy

Missing-value semantics remain fixed; fitted values are learned inside each candidate pipeline from its training rows (and later separately within every CV training fold).

| Pattern | Columns | Treatment |
|---|---|---|
| Structural categorical | `pumptype`, `drillmethod`, `piped_source`, `piped_pump` | Explicit `Not applicable` category |
| Other nominal/binary | Remaining nominal inputs | Explicit `Unknown` category |
| Structural numeric | `rehab_age` | Fill with 0 and retain `rehab_age_missing` |
| Moderate/high numeric missingness | `annual_rain`, `wpqty_wp`, `qtyhh_c_wp`, `improved_wponly_wp`, `wc_savings_wp`, `pumpstrokes` | Training-fitted median plus missing indicator |
| Low numeric missingness | Age, counts and location variables | Training-fitted median |

`pop_1000_missing` is retained separately because `pop_1000` may be log transformed. The four ordinal committee indices keep a separate missing code (`-1`). The holdout supplies no median or category frequency.

In [44]:
# Preview: rehab_age structural handling (also implemented inside FeatureEngineer, Section 8)
preview_rehab_age_missing = X_train["rehab_age"].isna().mean() * 100
print(f"rehab_age missing in training data: {preview_rehab_age_missing:.2f}%")

preview = X_train.groupby(X_train["rehabyn"], dropna=False)["rehab_age"].apply(lambda s: s.isna().mean() * 100)
print("\nrehab_age missing % by rehabyn (training data only):")
print(preview.round(2))

rehab_age missing in training data: 85.08%

rehab_age missing % by rehabyn (training data only):
rehabyn
No     100.00
Yes      0.47
NaN    100.00
Name: rehab_age, dtype: float64


The training-only preview confirms that `rehab_age` missingness is structural for water points marked as not rehabilitated. Its 0 fill and explicit indicator are created inside `FeatureEngineer`, so future CV folds receive the same semantics.

# 8. Feature Engineering and Ablation Design

`FeatureEngineer` computes six features without using target labels. **Fixed semantic indicators:** `rehab_age_missing`, `pop_1000_missing`. **Optional features:** `wp_age_group`, `was_rehabilitated`, `people_per_household`, `has_management_committee`. Base candidates omit the optional four at the `ColumnTransformer`; engineered candidates include them. The CatBoost-native family receives the same base-versus-engineered ablation as the other families; no benefit is assumed. We compare performance with and without optional engineered features later, using the same training CV folds, to learn whether they improve generalization rather than assuming they do.

`wp_age_group` uses age tertile edges and a median fitted only on rows passed to `.fit()`. `people_per_household` divides people by households; zero denominator and missing inputs yield missing, never infinity. `was_rehabilitated` maps Yes to 1, No to 0, and missing to missing. The management flag follows the prior Water committee definition. Target-derived aggregates such as country abandonment rates remain out of scope because of their elevated leakage risk.

In [45]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    # Adds engineered features. All fitted statistics come from the data passed
    # to .fit() only (i.e. the training split, when used inside the full pipeline).

    def fit(self, X, y=None):
        wp_age_train = X["wp_age"].astype(float)
        self.wp_age_median_ = wp_age_train.median()
        wp_age_filled = wp_age_train.fillna(self.wp_age_median_)
        q1, q2 = wp_age_filled.quantile([1 / 3, 2 / 3])
        self.wp_age_edges_ = (float(q1), float(q2))
        return self

    def transform(self, X):
        X = X.copy()

        # people_per_household: guard against division by zero / missing inputs
        qtyhh_safe = X["qtyhh_wp"].replace(0, np.nan)
        X["people_per_household"] = X["qtypeople_wp"] / qtyhh_safe

        # was_rehabilitated: clean 0/1/NaN mirror of rehabyn
        X["was_rehabilitated"] = X["rehabyn"].map({"Yes": 1, "No": 0})

        # has_management_committee: Water committee vs. everything else
        X["has_management_committee"] = (X["whomanage_wp"] == "Water committee").astype(int)

        # rehab_age: preserve structural meaning (fill 0 + explicit indicator)
        X["rehab_age_missing"] = X["rehab_age"].isna().astype(int)
        X["rehab_age"] = X["rehab_age"].fillna(0)

        # pop_1000: separate missing-indicator (kept apart from the log1p branch)
        X["pop_1000_missing"] = X["pop_1000"].isna().astype(int)

        # wp_age_group: bin using edges fitted on the training split only
        wp_age_filled = X["wp_age"].fillna(self.wp_age_median_)
        q1, q2 = self.wp_age_edges_
        X["wp_age_group"] = pd.cut(
            wp_age_filled,
            bins=[-np.inf, q1, q2, np.inf],
            labels=["New", "Established", "Old"],
        ).astype(str)

        return X


# Quick sanity preview, fit on the training split only
_fe_preview = FeatureEngineer().fit(X_train)
print(f"wp_age_group bin edges (fitted on X_train only): {_fe_preview.wp_age_edges_}")
_preview_out = _fe_preview.transform(X_train.head())
_preview_out[[
    "wp_age", "wp_age_group", "rehabyn", "was_rehabilitated",
    "whomanage_wp", "has_management_committee",
    "qtypeople_wp", "qtyhh_wp", "people_per_household",
    "rehab_age", "rehab_age_missing", "pop_1000", "pop_1000_missing",
]]

wp_age_group bin edges (fitted on X_train only): (2.0, 4.0)


,wp_age,wp_age_group,rehabyn,was_rehabilitated,whomanage_wp,has_management_committee,qtypeople_wp,qtyhh_wp,people_per_household,rehab_age,rehab_age_missing,pop_1000,pop_1000_missing
283,1.0,New,NaN,NaN,Water committee,1,11.0,2.0,5.500000,0.0,1,NaN,1
1203,2.0,New,No,0.0,Water committee,1,225.0,45.0,5.000000,0.0,1,1423.596305,0
518,1.0,New,No,0.0,Water committee,1,530.0,120.0,4.416667,0.0,1,NaN,1
1740,2.0,New,No,0.0,Water committee,1,174.0,29.0,6.000000,0.0,1,1262.005344,0
1755,2.0,New,No,0.0,Water committee,1,700.0,100.0,7.000000,0.0,1,1591.008797,0


# 9. Categorical Encoding Strategy

The four committee-capacity ratings retain the meaningful order Inadequate < Minimum < Moderate < Advanced, with a separate `-1` code for missing. Structural nominal fields use `Not applicable`; other nominal/binary fields use `Unknown`. Both one-hot branches use `handle_unknown="infrequent_if_exist"` and the same configurable `rare_min_frequency` (default 10). Candidate thresholds `[5, 10, 20]` may later be compared *inside training-only CV*. The threshold is a modelling choice, not a data-correctness rule.

The earlier training-only ordinal review remains below as historical context; no holdout labels enter that review.
For the CatBoost-native family, these nominal variables remain one categorical column each, with the same `Not applicable` and `Unknown` missing labels. The one-hot rare threshold does not apply to native categories. The four ordered committee indices remain numeric rather than nominal strings.

Transformed native categorical names use deterministic `cat_not_applicable__` and `cat_unknown__` prefixes, in structural-then-other group order. The metadata lists these exact DataFrame column names rather than asking Notebook 04 to infer them from dtype.


In [46]:
for col in ["wc_admin_index_wp", "wc_finance_index_wp", "wc_mgmt_index_wp", "wc_maint_index_wp"]:
    ct = pd.crosstab(X_train[col], y_train, normalize="index")[CLASS_ORDER] * 100
    print(f"--- {col} (training split only, row %) ---")
    print(ct.round(1))
    print()

--- wc_admin_index_wp (training split only, row %) ---
functional3        Functional  Partially functional  \
wc_admin_index_wp                                     
Advanced                 94.1                   0.0   
Inadequate               61.2                  11.5   
Minimum                  90.2                   5.9   
Moderate                 69.5                   9.2   

functional3        Abandoned or not functional  
wc_admin_index_wp                               
Advanced                                   5.9  
Inadequate                                27.3  
Minimum                                    3.9  
Moderate                                  21.2  

--- wc_finance_index_wp (training split only, row %) ---
functional3          Functional  Partially functional  \
wc_finance_index_wp                                     
Advanced                   82.4                   2.4   
Inadequate                 68.6                   7.9   
Minimum                    69.1   

The training-only table above shows mixed class distributions in each observed rating level, consistent with the earlier decision to retain these ordinal predictors. It cannot prove that a field is available before an operational prediction; that timing remains a domain assumption to review separately.

In [47]:
CAT_NOT_APPLICABLE_COLS = ["pumptype", "drillmethod", "piped_source", "piped_pump"]
CAT_UNKNOWN_RAW_COLS = [
    "country", "admin1", "wptype", "whomanage_wp", "rehabyn", "season",
    "cwfunded_wp", "wc_present_wp", "paytocollect_wp", "lockedfullday_wp",
]
ORDINAL_COLS = ["wc_admin_index_wp", "wc_finance_index_wp", "wc_mgmt_index_wp", "wc_maint_index_wp"]
ORDINAL_CATEGORY_ORDER = ["Inadequate", "Minimum", "Moderate", "Advanced", "Missing"]
RARE_MIN_FREQUENCY_CANDIDATES = [5, 10, 20]

def remap_ordinal_missing_code(codes):
    return np.where(codes == 4, -1, codes)

print("Structural categorical:", CAT_NOT_APPLICABLE_COLS)
print("Unknown categorical:", CAT_UNKNOWN_RAW_COLS)
print("Ordinal:", ORDINAL_COLS)

Structural categorical: ['pumptype', 'drillmethod', 'piped_source', 'piped_pump']
Unknown categorical: ['country', 'admin1', 'wptype', 'whomanage_wp', 'rehabyn', 'season', 'cwfunded_wp', 'wc_present_wp', 'paytocollect_wp', 'lockedfullday_wp']
Ordinal: ['wc_admin_index_wp', 'wc_finance_index_wp', 'wc_mgmt_index_wp', 'wc_maint_index_wp']


# 10. Model-Family Numerical Preprocessing

The **scale-sensitive** family (for example, later Logistic Regression) imputes numerics, optionally applies `log1p` to the non-negative skewed `wp_age`, `qtypeople_wp`, `qtyhh_wp`, and `pop_1000`, then applies `StandardScaler` to continuous numeric branches. Magnitudes affect its optimization or distance calculations.

The **tree/boosting** family (for example, later Random Forest, HistGradientBoostingClassifier, or XGBoost) uses the same cleaning, imputers, encoding, and missingness indicators. It omits scaling and log transformation by default because threshold splits generally do not need them. A log variant remains configurable for future training CV. Negative-capable columns such as `elevation_wp` are never logged. Binary semantic flags remain unscaled.
The **CatBoost-native** family reuses the tree family's numeric median imputation, missing indicators, and ordinal encoding. It preserves nominal categories as strings instead of applying `OneHotEncoder`, while leaving scaling and log transformation off. CatBoost can learn categorical representations internally when Notebook 04 fits it within each CV fold. This notebook does not recreate target statistics or fit CatBoost.


In [48]:
NUM_LOG_CANDIDATES = ["wp_age", "qtypeople_wp", "qtyhh_wp", "pop_1000"]
NUM_OTHER = ["elevation_wp", "latitude_wp", "longitude_wp", "rehab_age"]
NUM_WITH_INDICATOR = [
    "annual_rain", "wpqty_wp", "qtyhh_c_wp", "improved_wponly_wp",
    "wc_savings_wp", "pumpstrokes",
]
SEMANTIC_FLAGS = ["rehab_age_missing", "pop_1000_missing"]
OPTIONAL_NUM_WITH_INDICATOR = ["people_per_household", "was_rehabilitated"]
OPTIONAL_FLAGS = ["has_management_committee"]
OPTIONAL_CATEGORICAL = ["wp_age_group"]
print("Log candidates:", NUM_LOG_CANDIDATES)

Log candidates: ['wp_age', 'qtypeople_wp', 'qtyhh_wp', 'pop_1000']


# 11. Feature Set and Candidate Variant Definition

The 52 raw columns are accounted for: 1 target, 7 target-leakage exclusions, 10 identifier/survey/admin exclusions, 2 conservative pending-investigation exclusions, and **32 approved raw predictors**. `grant_number_wp` and `balance_any_dollars_wp` remain excluded pending separate investigation. Base/semantic variants use the 32 raw predictors and two required indicators; engineered/full variants add the four optional features.

`qtyhh_wp` and `qtypeople_wp` (correlation about 0.82) remain as raw predictors despite possible redundancy. Latitude/longitude also remain alongside broader geography despite possible location memorization. No supervised selector is fitted here. If `SelectKBest`, mutual information, or model-based selection is explored later, it must be inside each CV fold's complete pipeline, so validation labels cannot influence feature selection before scoring.

In [49]:
RAW_PREDICTOR_COLS = (
    CAT_NOT_APPLICABLE_COLS + CAT_UNKNOWN_RAW_COLS + ORDINAL_COLS
    + NUM_LOG_CANDIDATES + NUM_OTHER + NUM_WITH_INDICATOR
)
assert len(RAW_PREDICTOR_COLS) == 32
assert len(set(RAW_PREDICTOR_COLS)) == 32
assert set(RAW_PREDICTOR_COLS) == set(X_train.columns)
print("Approved raw predictor count:", len(RAW_PREDICTOR_COLS))

Approved raw predictor count: 32


# 12. Build Candidate Preprocessing Pipelines

Each builder call returns an **unfitted** sklearn pipeline: fixed cleaner → feature engineer → `ColumnTransformer`. The scale-sensitive and tree/boosting branches remain unchanged; a third branch preserves native categorical strings for CatBoost. Notebook 04 can append `("classifier", model)` to these steps and cross-validate the complete pipeline. `remainder="drop"` prevents accidental passage of unapproved or optional columns. All one-hot branches share the chosen rare threshold.

In [50]:
class CategoricalStringifier(BaseEstimator, TransformerMixin):
    """Keep imputed nominal columns as CatBoost-safe Python strings."""

    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        output = X.copy()
        for column in output.columns:
            output[column] = output[column].astype(str)
        return output

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        return np.asarray(input_features, dtype=object)


def catboost_categorical_feature_names(include_optional_engineered_features):
    """Exact transformed DataFrame names, in ColumnTransformer order."""
    unknown_cols = CAT_UNKNOWN_RAW_COLS + (OPTIONAL_CATEGORICAL if include_optional_engineered_features else [])
    return ([f"cat_not_applicable__{column}" for column in CAT_NOT_APPLICABLE_COLS]
            + [f"cat_unknown__{column}" for column in unknown_cols])


def build_catboost_native_pipeline(include_optional_engineered_features):
    """Reuse the existing tree numeric/ordinal branches; replace only nominal OHE."""
    tree_reference = build_preprocessing_pipeline(
        "tree_boosting", include_optional_engineered_features,
        rare_min_frequency=10, use_log_transform=False, scale_numeric=False,
    )
    tree_branches = tree_reference.named_steps["preprocessing"].transformers
    native_branches = []
    for name, transformer, columns in tree_branches:
        if name in {"cat_not_applicable", "cat_unknown"}:
            fill = "Not applicable" if name == "cat_not_applicable" else "Unknown"
            transformer = Pipeline([
                ("impute", SimpleImputer(strategy="constant", fill_value=fill)),
                ("to_strings", CategoricalStringifier()),
            ])
        native_branches.append((name, transformer, columns))
    preprocessor = ColumnTransformer(
        native_branches, remainder="drop", verbose_feature_names_out=True,
    )
    preprocessor.set_output(transform="pandas")
    return Pipeline([
        ("fixed_cleaning", FixedDataCleaner()),
        ("feature_engineering", FeatureEngineer()),
        ("preprocessing", preprocessor),
    ])


def build_preprocessing_pipeline(model_family, include_optional_engineered_features,
                                 rare_min_frequency=10, use_log_transform=None,
                                 scale_numeric=None):
    if model_family == "catboost_native":
        # The existing builder default is 10 for OHE families. It is ignored here;
        # explicit alternative rare thresholds are invalid for native categories.
        if rare_min_frequency not in (None, 10):
            raise ValueError("rare_min_frequency is not applicable to catboost_native")
        if use_log_transform not in (None, False) or scale_numeric not in (None, False):
            raise ValueError("CatBoost-native baseline does not use log transforms or scaling")
        return build_catboost_native_pipeline(include_optional_engineered_features)
    if model_family not in {"scale_sensitive", "tree_boosting"}:
        raise ValueError("model_family must be scale_sensitive, tree_boosting, or catboost_native")
    if not isinstance(rare_min_frequency, int) or rare_min_frequency < 1:
        raise ValueError("rare_min_frequency must be a positive integer")
    if use_log_transform is None:
        use_log_transform = model_family == "scale_sensitive"
    if scale_numeric is None:
        scale_numeric = model_family == "scale_sensitive"

    def categorical_branch(fill_value):
        return Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value=fill_value)),
            ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                      min_frequency=rare_min_frequency, sparse_output=False)),
        ])

    ordinal_branch = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("ordinal", OrdinalEncoder(categories=[ORDINAL_CATEGORY_ORDER] * len(ORDINAL_COLS))),
        ("remap_missing", FunctionTransformer(remap_ordinal_missing_code,
                                                 feature_names_out="one-to-one")),
    ])
    log_steps = [("impute", SimpleImputer(strategy="median"))]
    if use_log_transform:
        log_steps.append(("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")))
    if scale_numeric:
        log_steps.append(("scale", StandardScaler()))
    other_steps = [("impute", SimpleImputer(strategy="median"))]
    indicator_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
    if scale_numeric:
        other_steps.append(("scale", StandardScaler()))
        indicator_steps.append(("scale", StandardScaler()))

    unknown_cols = CAT_UNKNOWN_RAW_COLS + (OPTIONAL_CATEGORICAL if include_optional_engineered_features else [])
    indicator_cols = NUM_WITH_INDICATOR + (OPTIONAL_NUM_WITH_INDICATOR if include_optional_engineered_features else [])
    flags = SEMANTIC_FLAGS + (OPTIONAL_FLAGS if include_optional_engineered_features else [])
    preprocessor = ColumnTransformer([
        ("cat_not_applicable", categorical_branch("Not applicable"), CAT_NOT_APPLICABLE_COLS),
        ("cat_unknown", categorical_branch("Unknown"), unknown_cols),
        ("ordinal_wc_index", ordinal_branch, ORDINAL_COLS),
        ("num_skew", Pipeline(log_steps), NUM_LOG_CANDIDATES),
        ("num_other", Pipeline(other_steps), NUM_OTHER),
        ("num_indicator", Pipeline(indicator_steps), indicator_cols),
        ("semantic_flags", "passthrough", flags),
    ], remainder="drop", verbose_feature_names_out=True)
    preprocessor.set_output(transform="pandas")
    return Pipeline([
        ("fixed_cleaning", FixedDataCleaner()),
        ("feature_engineering", FeatureEngineer()),
        ("preprocessing", preprocessor),
    ])

VARIANTS = {
    "scaled_base": dict(model_family="scale_sensitive", include_optional_engineered_features=False,
                        rare_min_frequency=10, use_log_transform=True, scale_numeric=True),
    "scaled_engineered": dict(model_family="scale_sensitive", include_optional_engineered_features=True,
                              rare_min_frequency=10, use_log_transform=True, scale_numeric=True),
    "tree_base": dict(model_family="tree_boosting", include_optional_engineered_features=False,
                      rare_min_frequency=10, use_log_transform=False, scale_numeric=False),
    "tree_engineered": dict(model_family="tree_boosting", include_optional_engineered_features=True,
                            rare_min_frequency=10, use_log_transform=False, scale_numeric=False),
    "catboost_native_base": dict(model_family="catboost_native",
                                 include_optional_engineered_features=False,
                                 rare_min_frequency=None, use_log_transform=False,
                                 scale_numeric=False, one_hot_encode=False,
                                 native_categorical_handling=True,
                                 categorical_features=catboost_categorical_feature_names(False)),
    "catboost_native_engineered": dict(model_family="catboost_native",
                                       include_optional_engineered_features=True,
                                       rare_min_frequency=None, use_log_transform=False,
                                       scale_numeric=False, one_hot_encode=False,
                                       native_categorical_handling=True,
                                       categorical_features=catboost_categorical_feature_names(True)),
}
candidate_pipelines = {
    name: build_preprocessing_pipeline(**{
        key: value for key, value in config.items()
        if key not in {"one_hot_encode", "native_categorical_handling", "categorical_features"}
    })
    for name, config in VARIANTS.items()
}
print("Unfitted candidate variants:", list(candidate_pipelines))

Unfitted candidate variants: ['scaled_base', 'scaled_engineered', 'tree_base', 'tree_engineered', 'catboost_native_base', 'catboost_native_engineered']


# 13. Validate Candidate Preprocessing Pipelines

Each candidate is fitted on `X_train` only and transforms the locked `X_test` for **structural validation only**. No target or predictive metric is used. Different feature counts are valid because optional features and one-hot levels differ. CatBoost-native outputs remain mixed-type pandas DataFrames; categorical strings are checked separately from finite numeric values. These demonstration fits are not saved for later CV; Notebook 04 must instantiate fresh pipelines in each fold.

In [51]:
validation_rows = []
transformed_outputs = {}
for name, config in VARIANTS.items():
    pipeline = candidate_pipelines[name]
    train_out = pipeline.fit_transform(X_train)
    holdout_out = pipeline.transform(X_test)
    repeated = pipeline.transform(X_test)
    unseen_holdout = X_test.copy()
    unseen_holdout.loc[unseen_holdout.index[0], "country" if config["model_family"] == "catboost_native" else "wptype"] = (
        "__UNSEEN_COUNTRY_TEST__" if config["model_family"] == "catboost_native" else "__unseen_waterpoint_type__"
    )
    unseen_out = pipeline.transform(unseen_holdout)
    native = config["model_family"] == "catboost_native"
    categorical_features = config.get("categorical_features", [])
    numeric_train = train_out.drop(columns=categorical_features) if native else train_out
    numeric_holdout = holdout_out.drop(columns=categorical_features) if native else holdout_out
    numeric_unseen = unseen_out.drop(columns=categorical_features) if native else unseen_out
    train_values = numeric_train.to_numpy(dtype=float)
    holdout_values = numeric_holdout.to_numpy(dtype=float)
    native_valid = True
    if native:
        categorical_train = train_out[categorical_features]
        categorical_holdout = holdout_out[categorical_features]
        native_valid = (
            isinstance(train_out, pd.DataFrame) and isinstance(holdout_out, pd.DataFrame)
            and categorical_train.notna().all().all() and categorical_holdout.notna().all().all()
            and all(pd.api.types.is_object_dtype(dtype) or pd.api.types.is_categorical_dtype(dtype)
                    for dtype in categorical_train.dtypes)
            and all(categorical_train[column].map(lambda value: isinstance(value, str)).all()
                    for column in categorical_features)
            and all(categorical_holdout[column].map(lambda value: isinstance(value, str)).all()
                    for column in categorical_features)
            and unseen_out.loc[unseen_out.index[0], "cat_unknown__country"] == "__UNSEEN_COUNTRY_TEST__"
        )
    clean_holdout = pipeline.named_steps["fixed_cleaning"].transform(X_test)
    valid = (
        train_out.shape[0] == 1434 and holdout_out.shape[0] == 359
        and list(train_out.columns) == list(holdout_out.columns)
        and np.isfinite(train_values).all() and np.isfinite(holdout_values).all()
        and holdout_out.equals(repeated)
        and list(unseen_out.columns) == list(train_out.columns)
        and np.isfinite(numeric_unseen.to_numpy(dtype=float)).all()
        and native_valid
        and not clean_holdout["wc_savings_wp"].isin([999, 888]).any()
    )
    transformed_outputs[name] = (train_out, holdout_out)
    validation_rows.append({
        "variant": name, "model_family": config["model_family"],
        "optional_engineered_features": config["include_optional_engineered_features"],
        "scaling": config["scale_numeric"], "log_transform": config["use_log_transform"],
        "rare_min_frequency": config["rare_min_frequency"],
        "categorical_handling": "native strings" if native else "one-hot",
        "native_catboost_categories": native,
        "categorical_feature_count": len(categorical_features),
        "missing_categorical_values": (int(train_out[categorical_features].isna().sum().sum())
                                       + int(holdout_out[categorical_features].isna().sum().sum())) if native else 0,
        "train_rows": len(train_out), "holdout_rows": len(holdout_out),
        "output_feature_count": train_out.shape[1],
        "train_missing_after_transform": int(train_out.isna().sum().sum()),
        "holdout_missing_after_transform": int(holdout_out.isna().sum().sum()),
        "train_holdout_feature_name_parity": list(train_out.columns) == list(holdout_out.columns),
        "unseen_category_transform": list(unseen_out.columns) == list(train_out.columns)
                                      and (not native or native_valid),
        "status": "PASS" if valid else "FAIL",
    })
validation_summary = pd.DataFrame(validation_rows)
assert (validation_summary["status"] == "PASS").all(), validation_summary.to_string(index=False)
validation_summary

,variant,model_family,optional_engineered_features,scaling,log_transform,rare_min_frequency,categorical_handling,native_catboost_categories,categorical_feature_count,missing_categorical_values,train_rows,holdout_rows,output_feature_count,train_missing_after_transform,holdout_missing_after_transform,train_holdout_feature_name_parity,unseen_category_transform,status
0,scaled_base,scale_sensitive,False,True,True,10.0,one-hot,False,0,0,1434,359,102,0,0,True,True,PASS
1,scaled_engineered,scale_sensitive,True,True,True,10.0,one-hot,False,0,0,1434,359,110,0,0,True,True,PASS
2,tree_base,tree_boosting,False,False,False,10.0,one-hot,False,0,0,1434,359,102,0,0,True,True,PASS
3,tree_engineered,tree_boosting,True,False,False,10.0,one-hot,False,0,0,1434,359,110,0,0,True,True,PASS
4,catboost_native_base,catboost_native,False,False,False,NaN,native strings,True,14,0,1434,359,40,0,0,True,True,PASS
5,catboost_native_engineered,catboost_native,True,False,False,NaN,native strings,True,15,0,1434,359,46,0,0,True,True,PASS


# 14. Candidate Preprocessing Summary

**These rows are candidates, not rankings. No candidate is selected in Notebook 03.** Notebook 04 will pair suitable preprocessing variants with classifiers and compare them using training-only cross-validation on the same folds. Feature count is descriptive, not a quality score. The native CatBoost rows retain nominal strings and have no one-hot rare threshold (`N/A`). They are not predictive results.

In [52]:
validation_summary[["variant", "model_family", "optional_engineered_features",
                    "categorical_handling", "scaling", "log_transform",
                    "rare_min_frequency", "native_catboost_categories", "output_feature_count"]]


,variant,model_family,optional_engineered_features,categorical_handling,scaling,log_transform,rare_min_frequency,native_catboost_categories,output_feature_count
0,scaled_base,scale_sensitive,False,one-hot,True,True,10.0,False,102
1,scaled_engineered,scale_sensitive,True,one-hot,True,True,10.0,False,110
2,tree_base,tree_boosting,False,one-hot,False,False,10.0,False,102
3,tree_engineered,tree_boosting,True,one-hot,False,False,10.0,False,110
4,catboost_native_base,catboost_native,False,native strings,False,False,NaN,True,40
5,catboost_native_engineered,catboost_native,True,native strings,False,False,NaN,True,46


# 15. Persist Modern Workflow Handoff Data and Metadata

Only the frozen **raw** train/holdout predictor split, labels, and configuration metadata are saved under `data/processed/modern/`. The 32 approved raw predictors remain uncleaned in these CSV files so the cleaner stays inside later CV folds. No globally transformed matrix or fitted preprocessor is saved for CV. The final preprocessing + model artifact belongs after Notebook 05 selects and evaluates a complete pipeline. Legacy processed CSVs and `models/preprocessing_pipeline.joblib` are never written here.
The CatBoost metadata records its exact transformed categorical feature names and `null` for the inapplicable one-hot rare threshold. Notebook 04 must pass those names with the transformed pandas DataFrame when constructing CatBoost.


In [53]:
MODERN_DIR = Path("../data/processed/modern")
MODERN_DIR.mkdir(parents=True, exist_ok=True)
X_train.to_csv(MODERN_DIR / "X_train_raw.csv", index=False)
X_test.to_csv(MODERN_DIR / "X_test_raw.csv", index=False)
y_train.to_csv(MODERN_DIR / "y_train.csv", index=False, header=[TARGET])
y_test.to_csv(MODERN_DIR / "y_test.csv", index=False, header=[TARGET])
metadata = {
    "target": TARGET,
    "split": {"test_size": TEST_SIZE, "random_state": RANDOM_STATE, "stratify": TARGET},
    "raw_predictor_columns": list(X_train.columns),
    "rare_min_frequency_candidates": RARE_MIN_FREQUENCY_CANDIDATES,
    "variants": VARIANTS,
}
(MODERN_DIR / "preprocessing_variants.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print("Saved modern raw split and configuration metadata:")
for path in sorted(MODERN_DIR.iterdir()):
    print(" ", path)
print("Legacy preprocessing artifacts were NOT overwritten.")

Saved modern raw split and configuration metadata:
  ..\data\processed\modern\preprocessing_variants.json
  ..\data\processed\modern\X_test_raw.csv
  ..\data\processed\modern\X_train_raw.csv
  ..\data\processed\modern\y_test.csv
  ..\data\processed\modern\y_train.csv
Legacy preprocessing artifacts were NOT overwritten.


# 16. Preprocessing Validation

Every check below reports PASS/FAIL and a failed integrity condition raises an assertion. Only structural properties are checked; there is no classifier fit or predictive holdout calculation.

In [54]:
checks = {}
def check(name, condition):
    checks[name] = bool(condition)

check("Raw dataset SHA-256 unchanged", file_digest(DATA_PATH) == _raw_digest_before)
check("Raw in-memory copy unchanged", df_raw.equals(pd.read_excel(DATA_PATH)))
check("Target absent from predictors", TARGET not in X.columns)
check("All seven leakage columns absent", not set(LEAKAGE_COLS) & set(X.columns))
check("All ten identifier/admin columns absent", not set(IDENTIFIER_ADMIN_COLS) & set(X.columns))
check("Both pending-investigation columns absent", not set(PENDING_INVESTIGATION_COLS) & set(X.columns))
check("Exactly 32 approved raw predictors", len(X.columns) == 32 and set(X.columns) == set(RAW_PREDICTOR_COLS))
check("Locked split remains 1434/359", len(X_train) == 1434 and len(X_test) == 359)
check("Stratified class proportions within two percentage points", ((dist_train - dist_test).abs() < 2).all())
check("Sentinels converted by candidate cleaner", all(
    not pipe.named_steps["fixed_cleaning"].transform(X_train)["wc_savings_wp"].isin([999, 888]).any()
    for pipe in candidate_pipelines.values()))
check("Cleaner does not mutate input", X_train.equals(X_train_original) and
      all(pipe.named_steps["fixed_cleaning"].transform(X_train) is not X_train
          for pipe in candidate_pipelines.values()))
check("Age group edges fitted on training only", all(
    pipe.named_steps["feature_engineering"].wp_age_edges_ == _fe_preview.wp_age_edges_
    for pipe in candidate_pipelines.values()))
check("No target-derived feature", all(
    not any("abandonment_rate" in c or TARGET in c for c in pair[0].columns)
    for pair in transformed_outputs.values()))
for name, (train_out, holdout_out) in transformed_outputs.items():
    check(f"{name} train/holdout schema parity", list(train_out.columns) == list(holdout_out.columns))
    numeric_columns = [column for column in train_out.columns
                       if column not in VARIANTS[name].get("categorical_features", [])]
    check(f"{name} finite complete output", np.isfinite(train_out[numeric_columns].to_numpy(dtype=float)).all()
          and np.isfinite(holdout_out[numeric_columns].to_numpy(dtype=float)).all()
          and train_out.notna().all().all() and holdout_out.notna().all().all())
    check(f"{name} deterministic holdout transform", holdout_out.equals(candidate_pipelines[name].transform(X_test)))
check("Unseen category transforms in every candidate", validation_summary["unseen_category_transform"].all())
check("Base excludes optional engineered features", all(
    not any(f"__{name}" in col for col in transformed_outputs[variant][0].columns
            for name in OPTIONAL_CATEGORICAL + OPTIONAL_FLAGS + OPTIONAL_NUM_WITH_INDICATOR)
    for variant in ["scaled_base", "tree_base", "catboost_native_base"]))
check("Engineered variants include optional features", all(
    all(any(f"__{name}" in col for col in transformed_outputs[variant][0].columns)
        for name in OPTIONAL_CATEGORICAL + OPTIONAL_FLAGS + OPTIONAL_NUM_WITH_INDICATOR)
    for variant in ["scaled_engineered", "tree_engineered", "catboost_native_engineered"]))
check("Scale-sensitive candidate transformed", validation_summary.loc[validation_summary.model_family == "scale_sensitive", "status"].eq("PASS").all())
check("Tree/boosting candidate transformed", validation_summary.loc[validation_summary.model_family == "tree_boosting", "status"].eq("PASS").all())
check("CatBoost-native family builder exists", callable(build_catboost_native_pipeline)
      and isinstance(build_preprocessing_pipeline("catboost_native", False), Pipeline))
check("Both CatBoost-native candidates transform", validation_summary.loc[
    validation_summary.model_family == "catboost_native", "status"].eq("PASS").all())
for native_name in ["catboost_native_base", "catboost_native_engineered"]:
    native_config = VARIANTS[native_name]
    native_pipeline = candidate_pipelines[native_name]
    native_train, native_holdout = transformed_outputs[native_name]
    categorical_names = native_config["categorical_features"]
    preprocessor = native_pipeline.named_steps["preprocessing"]
    branch_steps = [step for _, branch, _ in preprocessor.transformers
                    if isinstance(branch, Pipeline) for _, step in branch.steps]
    check(f"{native_name} pandas output", isinstance(native_train, pd.DataFrame)
          and isinstance(native_holdout, pd.DataFrame))
    check(f"{native_name} categorical metadata exact", categorical_names == [
        column for column in native_train.columns if column.startswith("cat_not_applicable__")
        or column.startswith("cat_unknown__")])
    check(f"{native_name} categorical strings and no missing", all(
        native_train[column].map(lambda value: isinstance(value, str)).all()
        and native_holdout[column].map(lambda value: isinstance(value, str)).all()
        and native_train[column].notna().all() and native_holdout[column].notna().all()
        for column in categorical_names))
    check(f"{native_name} no OHE scaler or log", not any(
        isinstance(step, (OneHotEncoder, StandardScaler))
        or (isinstance(step, FunctionTransformer) and step.func is np.log1p)
        for step in branch_steps))
    check(f"{native_name} ordinal numeric with missing code", all(
        pd.api.types.is_numeric_dtype(native_train[f"ordinal_wc_index__{column}"])
        and set(native_train[f"ordinal_wc_index__{column}"].unique()) <= {-1.0, 0.0, 1.0, 2.0, 3.0}
        for column in ORDINAL_COLS))
    check(f"{native_name} semantic numeric flags", all(
        pd.api.types.is_numeric_dtype(native_train[f"semantic_flags__{column}"])
        for column in SEMANTIC_FLAGS))
    check(f"{native_name} unseen category preserved", validation_summary.set_index("variant").loc[
        native_name, "unseen_category_transform"])
    check(f"{native_name} sklearn clone succeeds", isinstance(clone(native_pipeline), Pipeline))
for native_name, tree_name in [
    ("catboost_native_base", "tree_base"),
    ("catboost_native_engineered", "tree_engineered"),
]:
    native_train, native_holdout = transformed_outputs[native_name]
    tree_train, tree_holdout = transformed_outputs[tree_name]
    numeric_names = [column for column in native_train.columns
                     if not column.startswith("cat_not_applicable__")
                     and not column.startswith("cat_unknown__")]
    check(f"{native_name} numeric parity with tree", all(
        native_part[numeric_names].equals(tree_part[numeric_names])
        for native_part, tree_part in [(native_train, tree_train),
                                       (native_holdout, tree_holdout)]))
check("Existing four shapes unchanged", all(
    transformed_outputs[name][0].shape == (1434, width)
    and transformed_outputs[name][1].shape == (359, width)
    for name, width in [("scaled_base", 102), ("scaled_engineered", 110),
                        ("tree_base", 102), ("tree_engineered", 110)]))
_code_sources = ["".join(c["source"]) for c in notebook_cells_for_validation[:37] if c["cell_type"] == "code"]
check("Holdout never passed to fit", not any(token in source for source in _code_sources
      for token in [".fit(X_test", ".fit_transform(X_test", ".fit( X_test", ".fit_transform( X_test"]))
check("No classifier or predictive metric invoked", not any(token in source for source in _code_sources
      for token in ["LogisticRegression(", "RandomForestClassifier(", "HistGradientBoostingClassifier(",
                    "XGBClassifier(", "CatBoostClassifier(", "f1_score(", "accuracy_score(", "confusion_matrix("]))
check("Six candidate variants in metadata", len(json.loads(
    (MODERN_DIR / "preprocessing_variants.json").read_text(encoding="utf-8"))["variants"]) == 6)
check("Modern raw handoff retains 32 columns", all(pd.read_csv(MODERN_DIR / p).shape[1] == 32
      for p in ["X_train_raw.csv", "X_test_raw.csv"]))
check("Legacy artifacts byte-identical", all(
    (file_digest(path) if os.path.exists(path) else None) == digest
    for path, digest in _legacy_digests_before.items()))
check_results = pd.DataFrame([{"check": name, "result": "PASS" if passed else "FAIL"}
                              for name, passed in checks.items()])
print(check_results.to_string(index=False))
print(f"{sum(checks.values())}/{len(checks)} checks passed")
assert all(checks.values()), "Preprocessing integrity validation failed"

                                                        check result
                                Raw dataset SHA-256 unchanged   PASS
                                 Raw in-memory copy unchanged   PASS
                                Target absent from predictors   PASS
                             All seven leakage columns absent   PASS
                      All ten identifier/admin columns absent   PASS
                    Both pending-investigation columns absent   PASS
                           Exactly 32 approved raw predictors   PASS
                                Locked split remains 1434/359   PASS
    Stratified class proportions within two percentage points   PASS
                     Sentinels converted by candidate cleaner   PASS
                                Cleaner does not mutate input   PASS
                      Age group edges fitted on training only   PASS
                                    No target-derived feature   PASS
                      scaled_base 

# 17. Final Preprocessing Summary and Modelling Handoff

### Fixed Data-Correctness Decisions

`functional3` remains a three-class target. The seven current-status leakage fields, ten identifier/survey/admin fields, and two pending-investigation fields are excluded. The locked stratified split uses 80/20 and seed 42. Every candidate starts with the same `FixedDataCleaner`, which converts only `wc_savings_wp` codes 999/888 to missing, and the same training-fitted `FeatureEngineer`. Structural categorical missingness becomes `Not applicable`; other nominal missingness becomes `Unknown`; `rehab_age` retains a zero fill plus indicator. No rows are deleted or outliers capped.

### Flexible Preprocessing Decisions

Optional engineered-feature inclusion, model family, scaling, and log transformation remain configuration choices. The one-hot rare threshold defaults to 10 (future candidates 5, 10, 20) for the scale-sensitive and tree/boosting families. It does not apply to CatBoost-native categories and is `null` in their metadata. No choice is made from holdout results.

### Three Candidate Preprocessing Families

**Scale-sensitive:** one-hot nominal encoding, median imputation, optional `log1p` on four non-negative skewed inputs, and `StandardScaler` on continuous inputs; intended for Logistic Regression and potentially SVM/KNN.

**Tree/boosting:** one-hot nominal encoding, the same required cleaning/imputation/indicators, and no scaling or log transformation by default; intended for Random Forest, HistGradientBoostingClassifier, and XGBoost.

**CatBoost native:** the same fixed cleaner, feature engineer, numeric median imputation/indicators, and numeric ordered committee indices as tree/boosting, but nominal columns remain one column each with safe string values. There is no `OneHotEncoder`, `StandardScaler`, log transformation, or one-hot rare grouping. CatBoost can process these categories internally when fitted later; this notebook does not import or fit CatBoost or compute target-derived encodings.

### Feature Engineering and Ablation

`rehab_age_missing` and `pop_1000_missing` are fixed semantic indicators. `wp_age_group`, `was_rehabilitated`, `people_per_household`, and `has_management_committee` remain optional. CatBoost receives the same base-versus-engineered comparison as the other families. Notebook 04 will compare each pair on the same development CV folds; no engineered-feature benefit is claimed here.

### Leakage Prevention and Handoff

The holdout never supplies fitted preprocessing statistics. During later model comparison, cleaner, engineer, imputer, encoder where applicable, and classifier must fit anew inside each CV training fold. `X_test` is transformed here only to verify structure; `y_test` is used only to verify split distribution. The modern handoff saves raw split CSVs and variant metadata, not a globally preprocessed CV matrix or fitted model. CatBoost must receive the transformed pandas DataFrame **and the exact categorical feature names in its variant metadata**; its primary baseline should use the native candidate, not the one-hot tree candidate.

### Future Modelling Handoff

Notebook 04 may pair Logistic Regression with `scaled_base`/`scaled_engineered`; Random Forest, HistGradientBoostingClassifier, and XGBoost with `tree_base`/`tree_engineered`; and CatBoost with `catboost_native_base`/`catboost_native_engineered`. No candidate is ranked here. Notebook 05 may later investigate CatBoost hyperparameters, class weighting or `auto_class_weights`, native numeric-missing handling, and optional engineered features. The locked holdout is reserved for one-time final evaluation. Planned primary metric: Macro-F1; supporting classification metrics remain Balanced Accuracy, Macro Precision/Recall, Weighted F1, per-class scores, and Confusion Matrix, with Log Loss or justified multiclass ROC-AUC when probability outputs are available.

### Unresolved Issues

`grant_number_wp` and `balance_any_dollars_wp` require separate investigation. Negative `elevation_wp` values need source verification. `qtyhh_wp`/`qtypeople_wp` may be redundant, and coordinates may encourage geographic memorization. The operational timing of committee indices should be reviewed. Target-derived aggregates remain unimplemented.
